# Get list of entrants

In [ ]:
tournament_slug = 'ngpr'
event_name = 'Melee Redemption'
event_name = 'Melee Singles'

In [51]:
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

In [52]:
%%bash
source ~/.bashrc

In [53]:
import os

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

In [54]:
transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)


In [55]:
# Define the API endpoint and query for tournament details
query = gql("""
query TournamentQuery($slug: String!) {
    tournament(slug: $slug) {
        id
        name
        city
        state
        countryCode
        startAt
        endAt
        events {
            id
            name
            numEntrants
        }
    }
}
""")

variables = {"slug": tournament_slug}

# Execute the query
try:
        tournament_result = client.execute(query, variable_values=variables)
        print(json.dumps(tournament_result, indent=2))
except Exception as e:
        print("Error fetching tournament details:", e)

{
  "tournament": {
    "id": 810907,
    "name": "New Game Plus Revival 8.14",
    "city": "Boston",
    "state": 3,
    "countryCode": "US",
    "startAt": 1753218000,
    "endAt": 1753242960,
    "events": [
      {
        "id": 1422769,
        "name": "Melee Redemption",
        "numEntrants": 10
      },
      {
        "id": 1422770,
        "name": "Project+ Singles (Free!)",
        "numEntrants": 7
      },
      {
        "id": 1422768,
        "name": "Melee Singles",
        "numEntrants": 28
      }
    ]
  }
}


In [56]:
# Extract the event id for "Melee Singles"
events = tournament_result['tournament']['events']
melee_singles_id = next((event['id'] for event in events if event_name in event['name']), None)
print(f"{event_name} Event ID: {melee_singles_id}")

Melee Redemption Event ID: 1422769


In [8]:
# Query to get entrants for Melee Singles
entrants_query = gql("""
query EventEntrants($eventId: ID!) {
    event(id: $eventId) {
        entrants(query: {page: 1, perPage: 512}) {
            nodes {
                id
                name
                participants {
                    gamerTag
                }
            }
        }
    }
}
""")

entrants_variables = {"eventId": melee_singles_id}

try:
        entrants_result = client.execute(entrants_query, variable_values=entrants_variables)
        print(json.dumps(entrants_result, indent=2))
except Exception as e:
        print("Error fetching entrants:", e)

{
  "event": {
    "entrants": {
      "nodes": [
        {
          "id": 20771889,
          "name": "nahome tesfay",
          "participants": [
            {
              "gamerTag": "nahome tesfay"
            }
          ]
        },
        {
          "id": 20771760,
          "name": "MAIF",
          "participants": [
            {
              "gamerTag": "MAIF"
            }
          ]
        },
        {
          "id": 20771740,
          "name": "sfy | sfy bees",
          "participants": [
            {
              "gamerTag": "sfy bees"
            }
          ]
        },
        {
          "id": 20771458,
          "name": "SIX | Embry",
          "participants": [
            {
              "gamerTag": "Embry"
            }
          ]
        },
        {
          "id": 20771311,
          "name": "CF | JAVI ON EARTH",
          "participants": [
            {
              "gamerTag": "JAVI ON EARTH"
            }
          ]
        },
        {
       

In [9]:
import polars as pl

# Extract entrant data and flatten participants' gamerTags

entrant_nodes = entrants_result['event']['entrants']['nodes']
data = []
for entrant in entrant_nodes:
    entrant_id = entrant['id']
    entrant_name = entrant['name']
    # There may be multiple participants per entrant; join their gamerTags with comma
    gamer_tags = [p['gamerTag'] for p in entrant.get('participants', [])]
    gamer_tag = ', '.join(gamer_tags)
    data.append({'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

entrants = pl.DataFrame(data)
entrants

id,name,gamertag
i64,str,str
20771889,"""nahome tesfay""","""nahome tesfay"""
20771760,"""MAIF""","""MAIF"""
20771740,"""sfy | sfy bees""","""sfy bees"""
20771458,"""SIX | Embry""","""Embry"""
20771311,"""CF | JAVI ON EARTH""","""JAVI ON EARTH"""
…,…,…
20766183,"""Downey""","""Downey"""
20762073,"""bfu""","""bfu"""
20759616,"""MOOP""","""MOOP"""


In [10]:
entrant_gamertags = entrants.select(pl.col('gamertag').str.to_lowercase()).to_series().to_list()

# collect seeding

In [11]:
player_ratings = pl.read_parquet('data/player-ratings.parquet')

In [24]:
pl.Config(tbl_rows=100)
entrant_ratings = (
    entrants
    .with_columns(pl.col('gamertag').str.to_lowercase())
    .join(
        player_ratings
            .with_columns(pl.col('tag').str.split(' | ').list.last().str.to_lowercase()),
        how='full',
        left_on='gamertag',
        right_on='tag',
    )
    #.filter(
    #    (pl.col('tag').str.to_lowercase() == pl.col('gamertag').str.to_lowercase()) |
    #    pl.col('gamertag').is_null()
    #)
    .filter(pl.col('gamertag').is_not_null())
    .sort('rating', descending=True)
    .group_by('id').first()
    .fill_null(0)
    .sort('rating', descending=True)
)
display(entrant_ratings)

id,name,gamertag,url,rating,tag
i64,str,str,str,f64,str
20770577,"""bonfire10""","""bonfire10""","""/league/nemelee/player/C9B8492…",42.003164,"""bonfire10"""
20768777,"""Ant""","""ant""","""/league/nemelee/player/5DBCC34…",35.631526,"""ant"""
20771274,"""hc | saucymain""","""saucymain""","""/league/nemelee/player/2407F41…",33.768498,"""saucymain"""
20769680,"""Sweat""","""sweat""","""/league/nemelee/player/DF3CE0B…",32.880512,"""sweat"""
20769115,"""Badboi""","""badboi""","""/league/nemelee/player/CDE68D1…",32.000816,"""badboi"""
20771267,"""zeldEx""","""zeldex""","""/league/nemelee/player/DBA78EE…",30.695035,"""zeldex"""
20768222,"""Gambit""","""gambit""","""/league/nemelee/player/62EEC11…",30.626982,"""gambit"""
20767219,"""Motobug""","""motobug""","""/league/nemelee/player/E12030B…",30.067921,"""motobug"""
20769573,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",29.015436,"""zaubermaus"""


In [25]:
entrant_seeding = (
    entrant_ratings
    .sort('rating', descending=True)
    .with_row_index('seed_num')
    .with_columns(pl.col('seed_num') + 1)
)
entrant_seeding

seed_num,id,name,gamertag,url,rating,tag
u32,i64,str,str,str,f64,str
1,20770577,"""bonfire10""","""bonfire10""","""/league/nemelee/player/C9B8492…",42.003164,"""bonfire10"""
2,20768777,"""Ant""","""ant""","""/league/nemelee/player/5DBCC34…",35.631526,"""ant"""
3,20771274,"""hc | saucymain""","""saucymain""","""/league/nemelee/player/2407F41…",33.768498,"""saucymain"""
4,20769680,"""Sweat""","""sweat""","""/league/nemelee/player/DF3CE0B…",32.880512,"""sweat"""
5,20769115,"""Badboi""","""badboi""","""/league/nemelee/player/CDE68D1…",32.000816,"""badboi"""
6,20771267,"""zeldEx""","""zeldex""","""/league/nemelee/player/DBA78EE…",30.695035,"""zeldex"""
7,20768222,"""Gambit""","""gambit""","""/league/nemelee/player/62EEC11…",30.626982,"""gambit"""
8,20767219,"""Motobug""","""motobug""","""/league/nemelee/player/E12030B…",30.067921,"""motobug"""
9,20769573,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",29.015436,"""zaubermaus"""


# Export seeding to start gg

In [14]:
# Query to get phase IDs for Melee Singles event
phases_query = gql("""
query EventPhases($eventId: ID!) {
    event(id: $eventId) {
        phases {
            id
            name
            numSeeds
        }
    }
}
""")

phases_variables = {"eventId": melee_singles_id}

try:
        phases_result = client.execute(phases_query, variable_values=phases_variables)
        print(json.dumps(phases_result, indent=2))
except Exception as e:
        print("Error fetching phases:", e)

phase_id = phases_result['event']['phases'][0]['id']
print(f'phase id: {phase_id}')

{
  "event": {
    "phases": [
      {
        "id": 2028888,
        "name": "Bracket",
        "numSeeds": 28
      }
    ]
  }
}
phase id: 2028888


In [15]:
# ...existing code...

# 1. Fetch seeds for the phase
seeds_query = gql("""
query PhaseSeeds($phaseId: ID!) {
  phase(id: $phaseId) {
    seeds(query: {perPage: 512}) {
      nodes {
        id
        entrant {
          id
        }
      }
    }
  }
}
""")
seeds_result = client.execute(seeds_query, variable_values={"phaseId": phase_id})
seed_nodes = seeds_result["phase"]["seeds"]["nodes"]
entrantid_to_seedid = {seed["entrant"]["id"]: seed["id"] for seed in seed_nodes}

# 2. Prepare seed mapping using correct seedId
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    entrant_id = int(row["id"])
    seed_id = entrantid_to_seedid.get(entrant_id)
    if seed_id:
        seed_mapping.append({
            "seedId": seed_id,
            "seedNum": row["seed_num"],
        })
    else:
        print(f"Warning: No seed found for entrant {entrant_id}")

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

Importing 26 seeds to phase 2028888...


In [16]:
entrant_seeding = (
    entrant_seeding
    .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))
)

/tmp/ipykernel_22710/1705157941.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))


In [17]:
# Prepare seed mapping from entrant_ratings for the phase
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    seed_mapping.append({
        "seedId": row["seed_id"],
        "seedNum": row['seed_num'],
    })

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

mutation = gql("""
mutation UpdatePhaseSeeding($phaseId: ID!, $seedMapping: [UpdatePhaseSeedInfo]!) {
  updatePhaseSeeding(phaseId: $phaseId, seedMapping: $seedMapping) {
    id
  }
}
""")

params = {
    "phaseId": phase_id,
    "seedMapping": seed_mapping,
}

try:
    result = client.execute(mutation, variable_values=params)
    print('Success!')
    print(result)
except Exception as e:
    print('Error:', e)

Importing 26 seeds to phase 2028888...
Error: {'message': 'Cannot modify seeds in started pools', 'extensions': {'category': 'validation'}, 'locations': [{'line': 2, 'column': 3}], 'path': ['updatePhaseSeeding'], 'type': 'validation', 'fields': None, 'fieldErrors': []}


# view previous tournaments by player

In [18]:
player = 'smoked_em'

In [19]:
from IPython.display import display, HTML

def display_side_by_side(*args, titles=('',)):
    html_str = ''
    if len(titles) > 0:
        html_str += '<div style="display:flex">'
    for df, title in zip(args, titles + ('',) * (len(args) - len(titles))):
        html_str += '<div style="margin-right:20px">'
        if title:
            html_str += f'<h2>{title}</h2>'
        html_str += df.to_html()
        html_str += '</div>'
    html_str += '</div>'
    display(HTML(html_str))

In [20]:
matches = pl.read_csv('data/matches-with-ratings.csv')

In [21]:
last_n_matches = (
    matches
    .filter(
        pl.col('winner').str.to_lowercase().str.contains(player.lower()) |
        pl.col('loser').str.to_lowercase().str.contains(player.lower())
    )
    .sort('tournament_date', 'encounter_id', descending=True)
    .head(40)
)
losses = (
    last_n_matches
    .filter(pl.col('loser').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('winner').alias('tag'),
        'winner_rating',
        'tournament_name',
        'tournament_date',
    )
    .sort('winner_rating', descending=False)
)
wins = (
    last_n_matches
    .filter(pl.col('winner').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('loser').alias('tag'),
        'loser_rating',
        'tournament_name',
        'tournament_date',
    )
    .sort('loser_rating', descending=True)
)
display_side_by_side(
    losses.to_pandas(),
    wins.to_pandas(),
    titles=('Losses', 'Wins')
)

,tag,winner_rating,tournament_name,tournament_date
,tag,loser_rating,tournament_name,tournament_date
